#### 2.3.4.2 RecursiveCharacterTextSplitter：最常用
文档切分器中较常用的是 `RecursiveCharacterTextSplitter (递归字符文本切分器) `，遇到 特定字符 时进行分割。默认情况下，它尝试进行切割的字符包括 ["\n\n", "\n", " ", ""] 。

RecursiveCharacterTextSplitter 代表了一类很典型的 RAG 切分思路：

优先按更 自然的文本边界 切分（使用切割的字符），若切分后的片段仍过大，再逐级退化到更细粒度的分隔符，以此类推。同一层里足够短的相邻片段，再按 chunk_size 与 chunk_overlap 合并为最终 chunk。

> 它的价值在于：尽量保留语义完整性，同时把片段控制在 embedding 与检索更友好的长度范围内。

此外，还可以通过 `separators` 参数自定义分割字符，例如添加“，”“。”（见后面的“中文文本的分隔符”）。

* 保留上下文：优先在自然语言边界（如段落、换行）处分割， 减少信息碎片化 。默认分隔符里没有句号，要在句子结尾处分割，需要自己添加标点。
* 智能分段：通过递归尝试多种分隔符，将文本分割为大小 不超过chunk_size 的片段（但不保证接近 chunk_size，见案例一的块 2）。
* 灵活适配：适用于多种文本类型（代码、Markdown、普通文本等），是LangChain中 最通用 的文本拆分器。

```python
RecursiveCharacterTextSplitter(
  self,
  separators: list[str] | None = None,
  keep_separator: bool | Literal['start', 'end'] = True,
  is_separator_regex: bool = False,
  kwargs: Any = {}
)
```

签名和当前源码（`langchain-text-splitters` 1.1.2，`langchain_text_splitters/character.py`）一致，只有最后一个参数写法不同：

> ⚠️ **与当前源码不同**：`kwargs: Any = {}` 应为 `**kwargs: Any`。`chunk_size`、`chunk_overlap`、`length_function`、`add_start_index`、`strip_whitespace` 都通过它交给父类 `TextSplitter`（含义见 2.3.3）。

| 参数 | 默认值 | 含义 |
|---|---|---|
| `separators` | `None`，即 `["\n\n", "\n", " ", ""]` | 分隔符列表，按顺序使用：先用第一个在文本里出现的分隔符切，切出的片段仍然太长，才对这个片段用后面的分隔符继续切。最后的 `""` 表示逐字符切 |
| `keep_separator` | `True`（父类默认是 `False`） | 含义同 2.3.4.1：`True` 等于 `"start"`，分隔符接在下一个片段的开头。默认分隔符都是空白字符，接在开头的会被 `strip_whitespace` 删掉，平时看不出来；换成中文标点时要改成 `"end"`（见后面） |
| `is_separator_regex` | `False` | `separators` 里的每一项是否按正则匹配 |
| `chunk_size`、`chunk_overlap` 等 | 4000、200 等 | 父类的参数。官方文档的说法：`chunk_size` 是块的最大长度，`chunk_overlap` 是相邻块的“目标重叠”（Target overlap） |

**返回值**和父类相同：`split_text` 返回 `list[str]`，`create_documents`、`split_documents` 返回 `list[Document]`（见 2.3.3）。

另外还有一个类方法 `from_language(language, **kwargs)`：按编程语言（Python、Markdown 等）换一套分隔符，用来切代码，属于按文档结构切分，后面学其他切分器时再看。


**原理：`_split_text` 递归切分**

`split_text(text)` 只有一行：`return self._split_text(text, self._separators)`。递归都在 `_split_text` 里（注释是我加的，省略了几行英文注释）：

```python
def _split_text(self, text: str, separators: list[str]) -> list[str]:
    final_chunks = []
    # ① 选分隔符：按顺序找第一个在 text 里出现的；排到 "" 就用 ""
    separator = separators[-1]
    new_separators = []
    for i, s_ in enumerate(separators):
        separator_ = s_ if self._is_separator_regex else re.escape(s_)
        if not s_:
            separator = s_
            break
        if re.search(separator_, text):
            separator = s_
            new_separators = separators[i + 1 :]   # 剩余分隔符：留给太长的片段继续切
            break

    # ② 用它切成片段（和 CharacterTextSplitter 用的是同一个函数）
    separator_ = separator if self._is_separator_regex else re.escape(separator)
    splits = _split_text_with_regex(text, separator_, keep_separator=self._keep_separator)

    # ③ 逐个处理片段
    good_splits = []
    separator_ = "" if self._keep_separator else separator   # 合并时用什么拼接（同 2.3.4.1 第 ③ 步）
    for s in splits:
        if self._length_function(s) < self._chunk_size:
            good_splits.append(s)                 # 短片段：暂存，等着和相邻的短片段合并
        else:
            if good_splits:                       # 长片段：先把暂存的短片段合并成块
                merged_text = self._merge_splits(good_splits, separator_)
                final_chunks.extend(merged_text)
                good_splits = []
            if not new_separators:
                final_chunks.append(s)            # 没有剩余分隔符：原样作为一块
            else:
                other_info = self._split_text(s, new_separators)   # 递归：用剩余分隔符切
                final_chunks.extend(other_info)
    # ④ 合并最后剩下的短片段
    if good_splits:
        merged_text = self._merge_splits(good_splits, separator_)
        final_chunks.extend(merged_text)
    return final_chunks
```

![RecursiveCharacterTextSplitter 的递归切分流程](assets/09-recursive-split-flow.svg)

和 2.3.4.1 的 `CharacterTextSplitter` 比，区别只在第 ③ 步怎么处理长片段：`CharacterTextSplitter` 只有一个分隔符，长片段原样成为一块并打印警告；`RecursiveCharacterTextSplitter` 换用剩余的、更细的分隔符继续切这个长片段。合并片段用的都是父类的 `_merge_splits`。

图右侧的四个要点，下面用案例一逐一对照。

案例一: 使用split_text()方法演示

In [1]:
# 1.导入相关依赖
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 2.定义RecursiveCharacterTextSplitter分割器对象
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=10,
    chunk_overlap=0,
    add_start_index=True,
)

# 3.定义拆分的内容
text="LangChain框架特性\n\n多模型集成(GPT/Claude)\n记忆管理功能\n链式调用设计。文档分析场景示例：需要处理PDF/Word等格式。"

# 4.拆分器分割
paragraphs = text_splitter.split_text(text)

for i,chunk in enumerate(paragraphs):
    print(f"块{i + 1},长度：{len(chunk)}")
    print(chunk)
    print('-' * 50)

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


块1,长度：10
LangChain框
--------------------------------------------------
块2,长度：3
架特性
--------------------------------------------------
块3,长度：9
多模型集成(GPT
--------------------------------------------------
块4,长度：8
/Claude)
--------------------------------------------------
块5,长度：6
记忆管理功能
--------------------------------------------------
块6,长度：9
链式调用设计。文档
--------------------------------------------------
块7,长度：10
分析场景示例：需要处
--------------------------------------------------
块8,长度：10
理PDF/Word等
--------------------------------------------------
块9,长度：3
格式。
--------------------------------------------------


**案例一的递归过程**

![案例一的递归过程](assets/10-recursive-example-tree.svg)

* 第 1 层：文本里有 `"\n\n"`，切成两个片段，都不短于 10，分别往下切；
* 第 2 层：“LangChain框架特性”里既没有 `"\n"` 也没有空格，这两个分隔符直接跳过，用 `""` 逐字符切出块 1、块 2（要点 1）。另一个片段用 `"\n"` 切成 4 段，其中两段短、两段长；
* 块 2“架特性”只有 3 个字，却没有和后面的内容合并：它们来自不同分支（要点 2）；
* 块 3、块 6 只有 9 个字：`keep_separator=True` 把换行留在片段开头，换行占了 1 个长度，合并后又被 `strip_whitespace` 删掉；
* 默认分隔符里没有“。”，最后一段只能逐字符切，块 6～块 9 都在句子中间断开。

另外，代码里设置了 `add_start_index=True`，但 `split_text` 只返回字符串，`start_index` 要用 `create_documents` 或 `split_documents` 才会写进 metadata（见 2.3.3），这里不起作用。

想看任意文本的递归过程，可以写一个子类，每次进入 `_split_text` 时打印这一层用哪个分隔符、切的是哪段文本（`_split_text` 是下划线开头的内部方法，这里只用来观察）：

In [2]:
class TracingSplitter(RecursiveCharacterTextSplitter):
    """每次进入 _split_text，打印这一层用哪个分隔符切、切的是哪段文本"""

    _depth = 0

    def _split_text(self, text, separators):
        # 和源码第 ① 步一样：第一个在 text 里出现的分隔符，排到 "" 就用 ""
        # （这里的分隔符不是正则，用 in 判断即可）
        sep = next((s for s in separators if s == "" or s in text), "")
        preview = text[:30] + ("…" if len(text) > 30 else "")
        print("    " * self._depth + f"[第 {self._depth + 1} 层，用 {sep!r} 切] {preview!r}（{len(text)} 字）")
        self._depth += 1
        try:
            return super()._split_text(text, separators)
        finally:
            self._depth -= 1


chunks = TracingSplitter(chunk_size=10, chunk_overlap=0).split_text(text)
print("\n结果和案例一相同：", chunks == paragraphs)

[第 1 层，用 '\n\n' 切] 'LangChain框架特性\n\n多模型集成(GPT/Claud…'（72 字）
    [第 2 层，用 '' 切] 'LangChain框架特性'（13 字）
    [第 2 层，用 '\n' 切] '\n\n多模型集成(GPT/Claude)\n记忆管理功能\n链式调…'（59 字）
        [第 3 层，用 '' 切] '\n多模型集成(GPT/Claude)'（18 字）
        [第 3 层，用 '' 切] '\n链式调用设计。文档分析场景示例：需要处理PDF/Word等…'（33 字）

结果和案例一相同： True


打印的就是上图中所有橙色的片段：只有长度不小于 `chunk_size` 的片段才会进入下一层。蓝色的短片段留在本层合并，不会出现在这里。

举例2：使用create_documents()方法演示，传入字符串列表，返回Document对象列表

In [6]:
# 1.导入相关依赖
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 2.打开.txt文件
with open("load/09-ai.txt", encoding="utf-8") as f:
    state_of_the_union = f.read()  #返回的是字符串

print(f"state_of_the_union的类型是:{type(state_of_the_union)}")
print(state_of_the_union)
# 3.定义RecursiveCharacterTextSplitter（递归字符分割器）
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20,
    #chunk_overlap=0,
    length_function=len
)

# 4.分割文本
texts = text_splitter.create_documents([state_of_the_union])

# 5.打印分割文本
for text in texts:
    print(f"⭐️{text.page_content}:len {len(text.page_content)}")

state_of_the_union的类型是:<class 'str'>
人工智能（AI）是什么？
人工智能（Artificial Intelligence，简称AI）是指由计算机系统模拟人类智能的技术，使其能够执行通常需要人类认知能力的任务，如学习、推理、决策和语言理解。AI的核心目标是让机器具备感知环境、处理信息并自主行动的能力。

1. AI的技术基础
AI依赖多种关键技术：

机器学习（ML）：通过算法让计算机从数据中学习规律，无需显式编程。例如，推荐系统通过用户历史行为预测偏好。

深度学习：基于神经网络的机器学习分支，擅长处理图像、语音等复杂数据。AlphaGo击败围棋冠军便是典型案例。

自然语言处理（NLP）：使计算机理解、生成人类语言，如ChatGPT的对话能力。

2. AI的应用场景
AI已渗透到日常生活和各行各业：

医疗：辅助诊断（如AI分析医学影像）、药物研发加速。

交通：自动驾驶汽车通过传感器和AI算法实现安全导航。

金融：欺诈检测、智能投顾（如风险评估模型）。

教育：个性化学习平台根据学生表现调整教学内容。

3. AI的挑战与未来
尽管前景广阔，AI仍面临问题：

伦理争议：数据隐私、算法偏见（如招聘AI歧视特定群体）。

就业影响：自动化可能取代部分人工岗位，但也会创造新职业。

技术瓶颈：通用人工智能（AGI）尚未实现，当前AI仅擅长特定任务。

未来，AI将与人类协作而非替代：医生借助AI提高诊断效率，教师利用AI定制课程。其发展需平衡技术创新与社会责任，确保技术造福全人类。
⭐️人工智能（AI）是什么？:len 12
⭐️人工智能（Artificial:len 15
⭐️Intelligence，简称AI）是指由计算机系统模拟人类智能的技术，使其能够执行通常需要人类认知能力的任务，如学习、推理、决策和语言理解。AI的核心目标是让机器具备感知环境、处理信息并自主行动的:len 99
⭐️让机器具备感知环境、处理信息并自主行动的能力。:len 23
⭐️1. AI的技术基础
AI依赖多种关键技术：

机器学习（ML）：通过算法让计算机从数据中学习规律，无需显式编程。例如，推荐系统通过用户历史行为预测偏好。:len 77
⭐️深度学习：基于神经网络的机器学习分支，擅长处理图像、语音等复杂数据。AlphaGo击败围棋冠军便是典

用同一个 `TracingSplitter` 看举例2 的递归过程：

In [4]:
_ = TracingSplitter(chunk_size=100, chunk_overlap=20).split_text(state_of_the_union)

[第 1 层，用 '\n\n' 切] '人工智能（AI）是什么？\n人工智能（Artificial I…'（634 字）
    [第 2 层，用 '\n' 切] '人工智能（AI）是什么？\n人工智能（Artificial I…'（131 字）
        [第 3 层，用 ' ' 切] '\n人工智能（Artificial Intelligence，…'（119 字）
            [第 4 层，用 '' 切] ' Intelligence，简称AI）是指由计算机系统模拟人…'（103 字）


只有第一段进入了更深的层，对照举例2 的输出：

* 第 1 层按 `"\n\n"` 切成段落。除第一段外，各段都短于 100，在第 1 层就合并成块（后 6 块），每块由几个完整的段落组成；
* 第一段 131 字，第 2 层按 `"\n"` 切出标题“人工智能（AI）是什么？”（第 1 块）和剩下的 119 字；
* 第 3 层：这 119 字里有空格（在“Artificial Intelligence”中间），于是按空格切，把英文术语拆到了两块里，第 2 块“人工智能（Artificial”只有 15 字；
* 第 4 层：剩下的 103 字已经没有更细的分隔符可用，只能逐字符切。第 3 块在“自主行动的”处断开，第 4 块开头的 20 个字和第 3 块末尾重复。这是整个结果里唯一的重叠：逐字符切时每个片段只有 1 个字，重叠才能正好凑够 20 字；段落级的块之间没有重叠，因为每块最后一个段落都超过 20 字（2.3.3：重叠由完整的片段组成）。

这里暴露了中文文本的两个问题：空格把英文术语切开、句子从中间断开。解决办法见后面的“中文文本的分隔符”。

举例4：使用split_documents()方法演示，利用PDFLoader加载文档，对文档的内容用递归切割器切割

In [11]:
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
import pymupdf
from langchain_pymupdf4llm import PyMuPDF4LLMLoader

pymupdf.set_messages(path=os.devnull)   # 关掉每页都会打印一次的解析提示

loader = PyMuPDF4LLMLoader("load/04-load.pdf", mode="single")

docs = loader.load()
# 4.定义切割器
text_splitter = RecursiveCharacterTextSplitter(
    #chunk_size=200,
    chunk_size=120,
    chunk_overlap=0,
    #chunk_overlap=100,
    length_function=len,
    add_start_index=True,
)

# 5.对pdf内容进行切割得到文档对象
paragraphs = text_splitter.split_documents(docs)

for para in paragraphs:
    print(para)
    print(f"Len={len(para.page_content)}")
    print('-------'*10)

page_content='**"他的车，他的命！** 他忽然想起来，一年，二年，至少有三四年；一滴汗，两滴汗，不 知道多少万滴汗，才挣出那辆车。从风里雨里的咬牙，从饭里茶里的自苦，才赚出那辆车。' metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2025-06-20T17:18:19+08:00', 'source': 'load/04-load.pdf', 'file_path': 'load/04-load.pdf', 'total_pages': 1, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-06-20T17:18:19+08:00', 'trapped': '', 'modDate': "D:20250620171819+08'00'", 'creationDate': "D:20250620171819+08'00'", 'start_index': 0}
Len=84
----------------------------------------------------------------------
page_content='那辆车是他的一切挣扎与困苦的总结果与报酬，像身经百战的武士的一颗徽章。……他老想 着远远的一辆车，可以使他自由，独立，像自己的手脚的那么一辆车。"' metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2025-06-20T17:18:19+08:00', 'source': 'load/04-load.pdf', 'file_path': 'load/04-load.pdf', 'total_pages': 1, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddat

* `mode="single"` 把整个 PDF 读成一个 `Document`（333 字）。`split_documents` 把它的 metadata 复制到每一块上，再加上 `start_index`（第 2 块是 85：第 1 块 84 字，中间的一个空格被 `strip_whitespace` 删掉了）。
* 切分点很奇怪，例如第 3 块结尾是“好梦想”，第 4 块开头是“的、利己的”，把“好梦想的”切成了两半。原因是 PDF 里的换行：PyMuPDF4LLM 把 PDF 每行末尾的换行变成了空格（输出里的“不 知道”“他老想 着”）。两个段落都超过 120 字，下一个可用的分隔符就是空格，于是切分点都落在 PDF 原来换行的位置，和句子无关。

中文 PDF 在项目里很常见，这个问题和举例2 一样，也是分隔符顺序造成的，见下一部分。

#### 中文文本的分隔符

官方文档的 Recursive 页面有一节专门讲这个问题（Splitting text from languages without word boundaries）：中文、日文、泰文的单词之间不用空格分开，用默认分隔符可能把词切断，所以建议在列表里加入标点：

```python
separators=[
    "\n\n", "\n", " ", ".", ",",
    "​",  # 零宽空格（泰文、日文等使用）
    "，",  # 全角逗号“，”
    "、",  # 顿号“、”
    "．",  # 全角句点“．”
    "。",  # 句号“。”
    "",
]
```

对中文来说，这个列表还有三个问题（下面的例子验证）：

1. 空格排在中文标点前面：文本里只要有空格（中英文混排的英文术语、PDF 换行变成的空格），就会先按空格切，举例2、举例4 的问题还在；
2. “，”“、”排在“。”前面：长段落会先在逗号处切开，块的边界落在句子中间；
3. `keep_separator` 默认是 `True`（`"start"`），标点会接在下一个片段的开头，块以“，”“。”开头。列表里也没有“！”“？”“；”。

所以中文文本可以按“段落 → 换行 → 句末标点 → 句内标点 → 空格 → 单字”的顺序排列，并设 `keep_separator="end"`，让标点留在句末。下面用举例2 的第一段（131 字，问题都出在这一段）比较三种设置：

In [6]:
with open("load/09-ai.txt", encoding="utf-8") as f:
    ai_text = f.read()
first_paragraph = ai_text.split("\n\n")[0]

OFFICIAL_SEPARATORS = ["\n\n", "\n", " ", ".", ",", "​", "，", "、", "．", "。", ""]
CHINESE_SEPARATORS = ["\n\n", "\n", "。", "！", "？", "；", "，", "、", " ", ""]

configs = {
    "1. 默认分隔符": dict(),
    "2. 官方给的分隔符": dict(separators=OFFICIAL_SEPARATORS),
    "3. 句末标点优先 + keep_separator='end'": dict(separators=CHINESE_SEPARATORS, keep_separator="end"),
}
for name, kwargs in configs.items():
    splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20, **kwargs)
    print(name)
    for chunk in splitter.split_text(first_paragraph):
        print(f"  {len(chunk):>3} 字 | {chunk[:12]!r} … {chunk[-12:]!r}")

1. 默认分隔符
   12 字 | '人工智能（AI）是什么？' … '人工智能（AI）是什么？'
   15 字 | '人工智能（Artific' … '能（Artificial'
   99 字 | 'Intelligence' … '境、处理信息并自主行动的'
   23 字 | '让机器具备感知环境、处理' … '理信息并自主行动的能力。'
2. 官方给的分隔符
   12 字 | '人工智能（AI）是什么？' … '人工智能（AI）是什么？'
   15 字 | '人工智能（Artific' … '能（Artificial'
   55 字 | 'Intelligence' … '常需要人类认知能力的任务'
   67 字 | '，使其能够执行通常需要人' … '理信息并自主行动的能力。'
3. 句末标点优先 + keep_separator='end'
   12 字 | '人工智能（AI）是什么？' … '人工智能（AI）是什么？'
   87 字 | '人工智能（Artific' … '、推理、决策和语言理解。'
   31 字 | 'AI的核心目标是让机器具' … '理信息并自主行动的能力。'


* 1：和举例2 一样，英文术语被空格拆开，第 3 块在句子中间断开；
* 2：官方的列表仍然先按空格切，“Artificial”和“Intelligence”还是在两块里；第 3 块在“，”处断开，第 4 块以“，”开头；
* 3：每块都在“？”或“。”处结束，英文术语保持完整。

第 3 种的块之间没有重叠：每个片段都是一整句，长度都超过 `chunk_overlap=20`，按 2.3.3 的规则就不会重叠。按句子切时，重叠要么设得比一句话长，要么就没有。

同样的设置用在举例4 的 PDF 上：

In [7]:
splitter = RecursiveCharacterTextSplitter(
    separators=CHINESE_SEPARATORS, keep_separator="end", chunk_size=120, chunk_overlap=0
)
for chunk in splitter.split_documents(docs):   # docs 是举例4 加载的 PDF
    content = chunk.page_content
    print(f"{len(content):>3} 字 | {content[:12]!r} … {content[-12:]!r}")

120 字 | '**"他的车，他的命！*' … '经百战的武士的一颗徽章。'
 38 字 | '……他老想 着远远的一辆' … '己的手脚的那么一辆车。"'
 71 字 | '**"他吃，他喝，他嫖，' … '烂，预备着到乱死岗子去。'
 98 字 | '……体面的、要强的、好梦' … '，个人主义的末路鬼！ "'


每块都在句末标点（加上后面的引号）处结束，不再从“好梦想的”中间切开。文本里换行变成的空格还在，它们只是不再被当作切分点；要去掉这些空格，得在加载之后另外清洗文本。

#### 在项目中使用

按做简历项目的需要，补充三点：

1. **从官方教程的参数起步**：官方教程（Build a semantic search engine、Build a RAG agent）用的都是 `RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)`，语义搜索教程还加了 `add_start_index=True`。本节例子里的 10、100、120 只是为了看清切分过程。
2. **中文要换算成 token 看**：`chunk_size` 默认按字符计，而模型（包括嵌入模型）的输入上限按 token 计。同样 1000 个字符，中文的 token 数大约是英文的 4 倍（下面用 tiktoken 估算）。照搬 1000 时，要对照所用嵌入模型的输入上限；也可以用 2.3.3 的 `from_tiktoken_encoder`、`from_huggingface_tokenizer` 直接按 token 计长度。
3. **用 `split_documents` 保留来源**：`split_text` 只返回字符串。用 `split_documents`（或 `create_documents`）时，每块的 metadata 带有文件来源 `source`，再加上 `add_start_index=True`，回答问题时就能给出引用出处（2.3.3）。中文文档再配上上一部分的分隔符和 `keep_separator="end"`。

In [8]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

with open("knowledge.txt", encoding="utf-8") as f:
    knowledge_text = f.read()

samples = {
    "中文：knowledge.txt": knowledge_text,
    "中文：09-ai.txt": ai_text,
    "英文：官方示例的开头": "Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. "
                           "Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.",
}
for name, sample in samples.items():
    ratio = len(enc.encode(sample)) / len(sample)
    print(f"{name}：每个字符约 {ratio:.2f} 个 token，1000 个字符约 {ratio * 1000:.0f} 个 token")

中文：knowledge.txt：每个字符约 0.86 个 token，1000 个字符约 862 个 token
中文：09-ai.txt：每个字符约 1.01 个 token，1000 个字符约 1011 个 token
英文：官方示例的开头：每个字符约 0.22 个 token，1000 个字符约 216 个 token


`cl100k_base` 是 OpenAI 模型的分词方式，其他嵌入模型的分词器不同，具体数字会变，但中文每个字符的 token 数远高于英文这一点是一样的。